This code provides a comprehensive and statistically more reliable evaluation of the Logistic Regression model's performance (accuracy in this case) compared to a single train-test split. By repeating the K-Fold process multiple times with different random shuffles, it minimizes the impact of a particular data split on the performance estimate, giving you greater confidence in your model's expected real-world accuracy and its stability. The inclusion of return_train_score=True also allows for a quick check for signs of overfitting.

In [20]:
import numpy as np
import pandas as pd4
from scipy.special import comb
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error,classification_report,make_scorer,confusion_matrix
from sklearn.model_selection import(
    train_test_split,
    KFold,
    LeaveOneOut,
    LeavePOut,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    RepeatedKFold
)

In [5]:
x,y = load_breast_cancer(return_X_y = True,as_frame = True)
y = y.map({1:0,0:1})
print(x.shape)
print(x.head())

(569, 30)
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  worst texture  worst perimeter  \
0 

In [6]:
y.value_counts()/len(y)

target
0    0.627417
1    0.372583
Name: count, dtype: float64

In [9]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.3,random_state = 0)
print(x_train.shape,x_test.shape)

(398, 30) (171, 30)


In [ ]:
logit = LogisticRegression(penalty = 'l2',solver = 'liblinear',C=10,random_state = 4,max_iter = 10000)
#l1 and l2 regression penalities, mostly we use l1 when we need to give importance to few columns, when we give importance to all columns - l2
#C defines how much strict penalty should be, 0.1  - high penalty, 10 - low penalty
#solver - liblinear method to train the model
kf = KFold (n_splits = 5,shuffle = True,random_state = 0)
clf = cross_validate(logit,x_train,y_train,cv = kf,scoring = 'accuracy',return_train_score=True)
clf['test_score']

array([0.9875    , 0.9625    , 0.925     , 0.92405063, 0.97468354])

logit = LogisticRegression(...): This creates an instance of the LogisticRegression classifier from scikit-learn. Logistic Regression is a linear model commonly used for binary classification tasks.

penalty = 'l2': This specifies the type of regularization to apply. 'l2' (L2 regularization or Ridge regularization) adds a penalty equal to the sum of the squared magnitudes of coefficients. This helps to prevent overfitting by discouraging large weights.

solver = 'liblinear': This specifies the algorithm used to optimize the logistic regression problem. 'liblinear' is a good choice for relatively small datasets and supports L1 and L2 penalties.

random_state = 4: This sets the seed for the random number generator used by the solver. Setting a random_state ensures that your results are reproducible; if you run the code again, you'll get the exact same results.

C = 10: This is the inverse of the regularization strength. Smaller values of C specify stronger regularization (more penalty). A value of 10 indicates relatively weak regularization compared to, say, 0.1.

max_iter = 10000: This sets the maximum number of iterations for the solver to converge. A high value like 10000 gives the solver plenty of chances to find a solution, especially for complex datasets.

K-Fold Cross-Validation: The dataset is first split into k (here, n_splits = 5) equal-sized "folds" or parts. The model is then trained k times. In each iteration, one fold is used as the validation (test) set, and the remaining k-1 folds are used as the training set.

Repeated: The entire K-Fold process is then repeated n_repeats (here, 10) times. Each repetition uses a different random shuffling of the data to create the folds.

n_splits = 5: Each repetition will perform 5-fold cross-validation.

n_repeats = 10: The 5-fold cross-validation process will be repeated 10 times.

random_state = 4: Ensures that the data splitting and shuffling for the folds are the same every time you run the code, making the results reproducible.

In [14]:
clf['train_score']

array([0.96540881, 0.9591195 , 0.97484277, 0.97178683, 0.96865204])

clf = cross_validate(...): This function from sklearn.model_selection orchestrates the cross-validation process.

logit: This is the estimator (your configured LogisticRegression model) that will be trained and evaluated.

x_train, y_train: These are your feature data (x_train) and target labels (y_train). It's assumed that these variables contain your training dataset, likely split from the original x, y loaded earlier.

cv = rkf: This tells cross_validate to use the RepeatedKFold strategy you just defined for splitting the data.

scoring = 'accuracy': This specifies the metric to use for evaluating the model's performance in each fold. 'accuracy' is a common metric for classification, representing the proportion of correctly predicted instances.

return_train_score=True: This is an important parameter. It tells cross_validate to not only calculate and return the scores on the test (validation) set for each fold but also the scores on the training set for each fold. This is very useful for diagnosing issues like overfitting (if train score is much higher than test score) or underfitting (if both are low).

In [15]:
print('mean train set accuracy',np.mean(clf['train_score']),np.std(clf['train_score']))
print('mean test set accuracy',np.mean(clf['test_score']),np.std(clf['test_score']))

mean train set accuracy 0.9679619881311489 0.005425110707151203
mean test set accuracy 0.9547468354430381 0.025913258297946374


**Pros of using K fold**

Better utilization of data compared to a single train/test split.

Provides a more robust estimate of model performance than a single split.

**Cons**

Can still be sensitive to the initial random shuffling of the data, especially if K is small or the dataset is small.

If the dataset has imbalanced classes, random splitting might result in some folds having very few or no samples of a minority class, leading to biased evaluation.

**When to use:** A widely used general-purpose cross-validation method for balanced datasets. Common values for K are 5 or 10.

In [19]:
#repeated K fold cross validation
logit = LogisticRegression(penalty = 'l2',solver = 'liblinear',random_state = 4,C = 10,max_iter = 10000)
rkf = RepeatedKFold(n_splits = 5,n_repeats=10,random_state = 4)
print('we expect k * n performance metrics: ',5*10)
clf = cross_validate(logit,x_train,y_train,cv = rkf,scoring = 'accuracy',return_train_score=True)

print('number of metrics obtained',len(clf['test_score']))

print('mean train set accuracy',np.mean(clf['train_score']),np.std(clf['train_score']))
print('mean test set accuracy',np.mean(clf['test_score']),np.std(clf['test_score']))

we expect k * n performance metrics:  50
number of metrics obtained 50
mean train set accuracy 0.9672745016856923 0.0072961432233506094
mean test set accuracy 0.9515094936708861 0.024498213076043808


Since there are n_splits = 5 folds in each repetition and n_repeats = 10 repetitions, the model will be trained and evaluated 5 * 10 = 50 times. For each of these 50 evaluations, you will get a performance metric (e.g., accuracy).

**Pros of using Repeated K fold**

Provides an even more robust and stable estimate of model performance by averaging across multiple shufflings. This reduces the impact of any particular lucky or unlucky data split.

Useful for estimating the variability of the model's performance (you get a mean and standard deviation of scores across K*N runs).

**Cons:**

Computationally more expensive than standard K-Fold because the model is trained N times as many times.

When to use: When you need a highly reliable and stable performance estimate, especially if the dataset is moderately sized or if you want to compare models with high confidence in their performance differences.

**When to use:** When you need a highly reliable and stable performance estimate, especially if the dataset is moderately sized or if you want to compare models with high confidence in their performance differences.

In [25]:
#leave one out cross validation - test for individual patients
logit = LogisticRegression(penalty = 'l2',solver = 'liblinear',C= 10,random_state = 4,max_iter = 10000)
loo = LeaveOneOut()
print('we expect n performance metrics as data in the train set: ',len(x_train))
clf = cross_validate(logit,x_train,y_train,cv = loo,scoring = 'accuracy',return_train_score=True)
print('number of metrics obtained',len(clf['test_score']))
print('mean train set accuracy',np.mean(clf['train_score']),np.std(clf['train_score']))
print('mean test set accuracy',np.mean(clf['test_score']),np.std(clf['test_score']))

we expect n performance metrics as data in the train set:  398
number of metrics obtained 398
mean train set accuracy 0.9652671417541105 0.0033268451073502456
mean test set accuracy 0.9522613065326633 0.21321282938268124


**Pros of using Leave One Out:**

Maximizes the amount of data used for training in each iteration (N-1 samples).

Provides a nearly unbiased estimate of the true error rate.

Every data point serves as a test point exactly once.

**Cons:**

Extremely computationally expensive for large datasets, as the model needs to be trained N times.

High variance in the error estimate, as the training sets in different folds are very similar.

**When to use:** Primarily for very small datasets where computational cost is less of a concern, and you need to maximize training data usage and minimize bias. Rarely used for large datasets due to its computational intensity.

In [26]:
#leave p out cross validation
logit  = LogisticRegression(penalty = 'l2',solver  = 'liblinear',random_state = 4,C = 10,max_iter = 10000)
lpo = LeavePOut(p=2)
x_train_small = x_train.head(100)
y_train_small = y_train.head(100)
clf = cross_validate(logit,x_train_small,y_train_small,cv = lpo,scoring = 'accuracy',return_train_score = True)
print('we expect ',comb(100,2),' metrics')
print('numer of metrics obtained',len(clf['test_score']))
print('mean train accuracy',np.mean(clf['train_score']),np.std(clf['train_score']))
print('mean test accuracy',np.mean(clf['test_score']),np.std(clf['test_score']))

we expect  4950.0  metrics
numer of metrics obtained 4950
mean train accuracy 0.9823314780457637 0.005020338686630207
mean test accuracy 0.9494949494949495 0.15561785450634855


**Pros of using leave P out:**

More exhaustive than K-Fold, potentially offering a more complete picture for small p.

**Cons:**

Even more computationally expensive than LOOCV. The number of combinations can grow astronomically fast (N choose p), making it impractical for most real-world datasets unless N is tiny and p is very small.

When to use: Very rarely used in practice due to its prohibitive computational cost. It's mostly of theoretical interest or for extremely niche applications with tiny datasets.

In [28]:
#stratified K fold cross validatation
logit = LogisticRegression(penalty = 'l2',solver = 'liblinear',C = 10,random_state= 4,max_iter = 10000)
skf = StratifiedKFold(n_splits = 5,shuffle = True,random_state = 0)
clf = cross_validate(logit,x_train,y_train,cv = skf,scoring = 'accuracy',return_train_score = True)

print('numer of metrics obtained',len(clf['test_score']))
print('mean train accuracy',np.mean(clf['train_score']),np.std(clf['train_score']))
print('mean test accuracy',np.mean(clf['test_score']),np.std(clf['test_score']))

numer of metrics obtained 5
mean train accuracy 0.9660830819581634 0.006068835658061979
mean test accuracy 0.9523417721518987 0.02147039375613738


**Pros of stratified K fold:**

Crucial for imbalanced datasets: Prevents a fold from having too few or no samples of a minority class, which could lead to a model that hasn't learned to predict that class or an unrepresentative evaluation.

Provides a more reliable and less biased estimate of performance for classification tasks compared to regular K-Fold, especially with class imbalance.

**Cons:**

Might not be suitable for regression tasks (as stratification is based on class labels).

When to use: Highly recommended for all classification problems, particularly when dealing with imbalanced class distributions. When you use cv=integer directly in scikit-learn's cross_val_score or GridSearchCV with a classifier, it often defaults to StratifiedKFold implicitly.

